# 03 — Gold: dim_policy

| Property | Value |
|----------|-------|
| **Gold Table** | `dim_policy` |
| **Grain** | One row per PolicyId |
| **Source** | `rpt.vwPolicy` |
| **PK** | `PolicyId` (int) |
| **Rows** | 20,057,914 |

> ⚠️ This is a large dimension (20M rows). PolicyId is confirmed unique.
> No versioning/SCD — each row is a distinct policy from 56 data sources.

**CONSTANT columns dropped**: AutoInvoice, AutoRenewal, IsWholeOrder, RetentionStructure,
SumInsuredCurrencyKey/Id, WillisPercentageOfOrder

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")

LAKEHOUSE = "The_Global_Loom"
TABLE = "dim_policy"
SOURCE_TABLE = "rpt.vwPolicy"

print(f"Config: {SOURCE_TABLE} → {LAKEHOUSE}.{TABLE}")

In [ ]:
# ============================================================
# Cell 2: Read silver source
# ============================================================
df_src = spark.table(SOURCE_TABLE)

print(f"Source: {df_src.count():,} rows × {len(df_src.columns)} cols")
df_src.printSchema()

## Cell 3: Transform

- Drop 7 CONSTANT columns (zero distinct values)
- Drop ETL audit columns
- Drop redundant Key/Id pairs (keep GlobalIds for FK consistency)
- Add DateKey columns (YYYYMMDD integers) for InceptionDate and ExpiryDate

In [ ]:
# ============================================================
# Cell 3: Transform
# ============================================================

# OPTIMIZED: Removed 5 high-cardinality/denormalized columns for Direct Lake performance:
#   - PolicyKey (string) - redundant with PolicyId (int)
#   - PolicyReference (10.9M distinct) - high cardinality text, rarely used in analytics
#   - PolicyDescription (3.1M distinct) - high cardinality text, rarely used
#   - SegmentCode - denormalized from dim_financial_segment
#   - OwnershipOrganisation - 8,929 distinct values, low analytical value
# Result: 30 → 25 columns, ~2-3 GB memory savings for 20M rows

df_clean = df_src.select(
    # --- Keys ---
    F.col("PolicyId").cast("int"),
    F.col("DataSourceInstanceId").cast("int"),

    # --- Dimension FKs ---
    F.col("GlobalFinancialGeographyId").cast("int"),
    F.col("GlobalFinancialSegmentId").cast("int"),
    F.col("GlobalLegalEntityId").cast("int"),
    F.col("GlobalCurrencyCode").cast("string"),

    # --- Dates ---
    F.col("InceptionDate").cast("timestamp"),
    F.col("ExpiryDate").cast("timestamp"),
    F.col("FirstInceptionDate").cast("timestamp"),
    F.col("RenewalDate").cast("timestamp"),
    F.col("PolicyIssuedDate").cast("timestamp"),

    # --- DateKeys (YYYYMMDD integers for dim_date join) ---
    F.date_format(F.col("InceptionDate"), "yyyyMMdd").cast("int").alias("InceptionDateKey"),
    F.date_format(F.col("ExpiryDate"), "yyyyMMdd").cast("int").alias("ExpiryDateKey"),

    # --- Classification ---
    F.col("RefInsuranceType").cast("string"),
    F.col("RefPolicyStatus").cast("string"),
    F.col("OpportunityType").cast("string"),

    # --- Financials ---
    F.col("AnnualizedCommission").cast("decimal(18,4)"),
    F.col("AnnualizedPremium").cast("decimal(18,4)"),
    F.col("SumInsured").cast("decimal(18,4)"),

    # --- Renewal chain ---
    F.col("RenewedFromPolicyId").cast("int"),

    # --- Flags ---
    F.col("IsRenewable").cast("boolean"),
    F.col("PolicyIssued").cast("boolean"),
    F.col("IsDeleted").cast("boolean")
)

# Filter out deleted Unknown members (ID = -1 AND IsDeleted = True)
before_count = df_clean.count()
df_clean = df_clean.filter(~((F.col("PolicyId") == -1) & (F.col("IsDeleted") == True)))
after_count = df_clean.count()

if before_count > after_count:
    print(f"Filtered out {before_count - after_count} deleted Unknown member(s) (ID=-1, IsDeleted=True)")

print(f"After column select + filter: {after_count:,} rows × {len(df_clean.columns)} cols (optimized from 30 to 25)")
print(f"   Dropped {len(df_src.columns) - len(df_clean.columns) + 2} source columns (+ 2 derived DateKeys)")


In [ ]:
# ============================================================
# Cell 4: Add Unknown member
# ============================================================
from pyspark.sql import Row

# Updated schema after optimization (removed PolicyKey, PolicyReference, PolicyDescription, SegmentCode, OwnershipOrganisation)
unknown_data = [(
    -1, -1,                                            # PolicyId, DSI
    -1, -1, -1, "Unknown",                             # Geo, Seg, Legal, Currency
    None, None, None, None, None,                      # Dates (null)
    None, None,                                        # DateKeys (null)
    "Unknown", "Unknown", "Unknown",                   # Classification
    None, None, None,                                  # Financials (null)
    None,                                              # RenewedFromPolicyId
    False, False, False                                # Flags
)]

unknown_row = spark.createDataFrame(unknown_data, schema=df_clean.schema)
df_final = df_clean.unionByName(unknown_row)

print(f"Added Unknown member: {df_final.count():,} rows")


In [ ]:
# ============================================================
# Cell 5: Data quality checks
# ============================================================
total = df_final.count()
dupes = total - df_final.select("PolicyId").distinct().count()
nulls = df_final.filter(F.col("PolicyId").isNull()).count()
has_unknown = df_final.filter(F.col("PolicyId") == -1).count()

# Check DateKey derivation
sample_datkey = df_final.filter(
    F.col("InceptionDate").isNotNull() & (F.col("PolicyId") != -1)
).select("InceptionDate", "InceptionDateKey").first()

print(f"DQ Checks")
print(f"   Total rows:     {total:,}")
print(f"   Duplicate PKs:  {dupes}")
print(f"   Null PKs:       {nulls}")
print(f"   Unknown member: {has_unknown} (expected 1)")
if sample_datkey:
    print(f"   DateKey sample: {sample_datkey['InceptionDate']} → {sample_datkey['InceptionDateKey']}")

assert dupes == 0, f"ERROR: Duplicates!"
print("\nAll DQ checks passed")

In [ ]:
# ============================================================
# Cell 6: Write to gold lakehouse
# ============================================================
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)

print(f"Written: {TABLE}")
print(f"   Rows: {spark.table(TABLE).count():,}")